# Feature Engineering

This notebook converts cleaned email text into machine learning features.

Goals:
- Encode target labels
- Split training and testing data
- Convert text into numerical vectors
- Understand sparse matrix representation
- Prepare features for ML model training

This phase bridges NLP preprocessing and machine learning.

In [123]:
# =========================================================
# IMPORT LIBRARIES
# =========================================================

import pandas as pd

from sklearn.model_selection import train_test_split

from sklearn.preprocessing import LabelEncoder

from sklearn.feature_extraction.text import TfidfVectorizer

from scipy.sparse import hstack

from sklearn.preprocessing import StandardScaler

In [124]:
# =========================================================
# LOAD CLEANED DATASET
# =========================================================

DATA_PATH = (

    "../data/interim/"
    "emails_cleaned.csv"
)

df = pd.read_csv(

    DATA_PATH,

    keep_default_na=False
)

print(

    df.shape
)

df.head()

(5755, 10)


,label,subject,from,to,date,content_type,body,email_length,clean_text,clean_text_length
0,ham,Re: New Sequences Window,Robert Elz <kre@munnari.OZ.AU>,Chris Garrigues <cwg-dated-1030377287.06fa6d@D...,"Thu, 22 Aug 2002 18:26:25 +0700","text/plain; charset=""us-ascii""","Date: Wed, 21 Aug 2002 10:54:46 -0500\n...",1598,date wed aug chri garrigu messageid cant repro...,808
1,ham,[zzzzteana] RE: Alexander,Steve Burt <Steve_Burt@cursor-system.com>,"""'zzzzteana@yahoogroups.com'"" <zzzzteana@yahoo...","Thu, 22 Aug 2002 12:46:18 +0100","text/plain; charset=""US-ASCII""","Martin A posted:\nTassos Papadopoulos, the Gre...",894,martin post tasso papadopoulo greek sculptor b...,399
2,ham,[zzzzteana] Moscow bomber,Tim Chapman <timc@2ubh.com>,zzzzteana <zzzzteana@yahoogroups.com>,"Thu, 22 Aug 2002 13:52:38 +0100","text/plain; charset=""US-ASCII""",Man Threatens Explosion In Moscow \n\nThursday...,1746,man threaten explos moscow thursday august pm ...,902
3,ham,[IRR] Klez: The Virus That Won't Die,Monty Solomon <monty@roscom.com>,undisclosed-recipient:;,"Thu, 22 Aug 2002 09:15:25 -0400","text/plain; charset=""us-ascii""",Klez: The Virus That Won't Die\n \nAlready the...,1125,klez viru wont die alreadi prolif viru ever kl...,590
4,ham,Re: [zzzzteana] Nothing like mama used to make,Stewart Smith <Stewart.Smith@ee.ed.ac.uk>,zzzzteana@yahoogroups.com,"Thu, 22 Aug 2002 14:38:22 +0100","text/plain; charset=""US-ASCII""","> in adding cream to spaghetti carbonara, whi...",1047,ad cream spaghetti carbonara effect pasta make...,464


In [125]:
# =========================================================
# CHECK MISSING VALUES
# =========================================================

df.isnull().sum()

label                0
subject              0
from                 0
to                   0
date                 0
content_type         0
body                 0
email_length         0
clean_text           0
clean_text_length    0
dtype: int64

In [126]:
# =========================================================
# LABEL DISTRIBUTION
# =========================================================

df["label"].value_counts()

label
ham     3895
spam    1860
Name: count, dtype: int64

# Label Encoding

Machine learning models cannot directly understand text labels.

We convert:
- ham  → 0
- spam → 1

This transforms categorical targets into numerical labels.

In [127]:
# =========================================================
# LABEL ENCODING
# =========================================================

label_encoder = LabelEncoder()

df["label_encoded"] = (

    label_encoder.fit_transform(
        df["label"]
    )
)

df[

    [
        "label",
        "label_encoded"
    ]

].head()

,label,label_encoded
0,ham,0
1,ham,0
2,ham,0
3,ham,0
4,ham,0


In [128]:
# =========================================================
# DEFINE FEATURES AND TARGET
# =========================================================

X = df["clean_text"]

y = df["label_encoded"]

print(

    X.shape,
    y.shape
)

(5755,) (5755,)


# Train-Test Split

The dataset is divided into:
- training data
- testing data

Training data teaches the model patterns.

Testing data evaluates how well the model generalizes to unseen emails.

In [129]:
# =========================================================
# TRAIN TEST SPLIT
# =========================================================

X_train, X_test, y_train, y_test = train_test_split(

    X,
    y,

    test_size=0.2,

    random_state=42,

    stratify=y
)

print(

    "X_train:",
    X_train.shape
)

print(

    "X_test:",
    X_test.shape
)

print(

    "y_train:",
    y_train.shape
)

print(

    "y_test:",
    y_test.shape
)

X_train: (4604,)
X_test: (1151,)
y_train: (4604,)
y_test: (1151,)


In [130]:
# =========================================================
# TRAIN LABEL DISTRIBUTION
# =========================================================

print(

    y_train.value_counts(
        normalize=True
    )
)

# =========================================================
# TEST LABEL DISTRIBUTION
# =========================================================

print(

    y_test.value_counts(
        normalize=True
    )
)

label_encoded
0    0.676803
1    0.323197
Name: proportion, dtype: float64
label_encoded
0    0.676803
1    0.323197
Name: proportion, dtype: float64


# TF-IDF Vectorization

TF-IDF converts text into numerical vectors.

TF (Term Frequency):
Measures how often a word appears in an email.

IDF (Inverse Document Frequency):
Reduces importance of extremely common words.

TF-IDF helps machine learning models identify important spam-related patterns.

In [131]:
# =========================================================
# TF-IDF VECTORIZATION
# =========================================================

vectorizer = TfidfVectorizer(

    max_features=5000,

    ngram_range=(1, 2),

    min_df=2,

    sublinear_tf=True
)

X_train_tfidf = vectorizer.fit_transform(

    X_train
)

X_test_tfidf = vectorizer.transform(

    X_test
)

print(

    "X_train_tfidf shape:",
    X_train_tfidf.shape
)

print(

    "X_test_tfidf shape:",
    X_test_tfidf.shape
)

X_train_tfidf shape: (4604, 5000)
X_test_tfidf shape: (1151, 5000)


In [132]:
# =========================================================
# SPARSE MATRIX ANALYSIS
# =========================================================

print(

    type(X_train_tfidf)
)

print(

    "Non-zero values:",
    X_train_tfidf.nnz
)

<class 'scipy.sparse._csr.csr_matrix'>
Non-zero values: 367991


# Sparse Matrix Concept

Most emails only contain a small subset of the total vocabulary.

As a result:
- most vector positions become zero
- only a few positions contain values

This creates a sparse matrix representation,
which is memory efficient for NLP tasks.

In [133]:
# =========================================================
# VOCABULARY SIZE
# =========================================================

vocab_size = len(

    vectorizer.vocabulary_
)

print(

    f"Vocabulary Size: {vocab_size}"
)

Vocabulary Size: 5000


In [134]:
# =========================================================
# SAMPLE VOCABULARY
# =========================================================

sample_vocab = list(

    vectorizer.vocabulary_.items()

)[:20]

sample_vocab

[('hi', np.int64(2029)),
 ('upgrad', np.int64(4648)),
 ('last', np.int64(2432)),
 ('week', np.int64(4824)),
 ('messag', np.int64(2745)),
 ('definit', np.int64(1065)),
 ('dcc', np.int64(1021)),
 ('databas', np.int64(996)),
 ('arent', np.int64(251)),
 ('mark', np.int64(2669)),
 ('spamassassin', np.int64(4130)),
 ('connect', np.int64(863)),
 ('via', np.int64(4717)),
 ('spamc', np.int64(4138)),
 ('detect', np.int64(1100)),
 ('run', np.int64(3810)),
 ('use', np.int64(4662)),
 ('known', np.int64(2405)),
 ('featur', np.int64(1591)),
 ('anyth', np.int64(207))]

In [135]:
# =========================================================
# FEATURE NAMES
# =========================================================

feature_names = vectorizer.get_feature_names_out()

feature_names[:30]

array(['aa', 'aaa', 'aab', 'ab', 'aba', 'abacha', 'abandon', 'abb', 'abc',
       'abf', 'abil', 'abl', 'absolut', 'absolut free', 'absolut legal',
       'absorb', 'abus', 'abus control', 'ac', 'aca', 'academ', 'acceler',
       'accept', 'accept apolog', 'access', 'access number', 'accessori',
       'accid', 'accommod', 'accomplish'], dtype=object)

In [136]:
# =========================================================
# TF-IDF WITHOUT FEATURE LIMIT
# =========================================================

full_vectorizer = TfidfVectorizer(

    ngram_range=(1, 2)
)

full_vectorizer.fit(

    X_train
)

full_vocab_size = len(

    full_vectorizer.vocabulary_
)

print(

    f"Full Vocabulary Size: {full_vocab_size}"
)

Full Vocabulary Size: 357158


In [137]:
# =========================================================
# SPARSE MATRIX DENSITY
# =========================================================

total_values = (

    X_train_tfidf.shape[0]
    *
    X_train_tfidf.shape[1]
)

non_zero_values = X_train_tfidf.nnz

density = (

    non_zero_values
    / total_values
)

print(

    f"Matrix Density: {density:.6f}"
)

Matrix Density: 0.015986


In [138]:
# =========================================================
# CREATE ARTIFACTS DIRECTORY
# =========================================================

import os

os.makedirs(

    "../artifacts",

    exist_ok=True
)

In [139]:
# =========================================================
# EXPORT TF-IDF VECTORIZER
# =========================================================

import joblib

output_path = (

    "../artifacts/"
    "tfidf_vectorizer.pkl"
)

joblib.dump(

    vectorizer,
    output_path
)

print(

    f"Vectorizer saved to:\n{output_path}"
)

Vectorizer saved to:
../artifacts/tfidf_vectorizer.pkl


In [140]:
# =========================================================
# EXPORT LABEL ENCODER
# =========================================================

label_encoder_path = (

    "../artifacts/"
    "label_encoder.pkl"
)

joblib.dump(

    label_encoder,
    label_encoder_path
)

print(

    f"Label encoder saved to:\n{label_encoder_path}"
)

Label encoder saved to:
../artifacts/label_encoder.pkl


# N-Grams

N-grams help the model learn word sequences.

Examples:
- unigram → "free"
- bigram  → "free money"

Bigram features often improve spam classification performance.

In [141]:
print(

    label_encoder.classes_
)

['ham' 'spam']


# Why We Save Artifacts

Machine learning systems must use the exact same preprocessing
during training and inference.

We export:
- TF-IDF vectorizer
- Label encoder

so that:
- Streamlit apps
- APIs
- deployed systems

can transform incoming emails consistently.

# =========================================================
# ADVANCED FEATURE ENGINEERING
# =========================================================

## Objective

In this section, we create advanced engineered features
that capture behavioral and structural spam patterns.

Unlike TF-IDF, which learns textual frequency patterns,
these handcrafted features represent real-world
email threat intelligence signals.

These features help the model detect:
- spam campaigns
- phishing behavior
- promotional content
- suspicious formatting
- obfuscated malicious emails

---

## Engineered Features

### 1. URL Count
Counts hyperlinks inside emails.

Spam emails often contain:
- phishing links
- tracking URLs
- malicious redirects

---

### 2. Exclamation Count
Measures aggressive punctuation usage.

Spam emails frequently use:
- urgency
- emotional triggers
- marketing pressure

Examples:
- FREE!!!
- BUY NOW!!!
- LIMITED OFFER!!!

---

### 3. Uppercase Ratio
Measures excessive capital letter usage.

Spam messages often contain:
- all-caps marketing text
- attention-grabbing phrases

Examples:
- WIN MONEY NOW
- URGENT RESPONSE REQUIRED

---

### 4. HTML Tag Count
Counts embedded HTML tags.

Many spam campaigns use:
- HTML email templates
- hidden content
- tracking pixels
- styled advertisements

---

### 5. Special Character Count
Measures unusual symbol density.

Obfuscated spam often includes:
- excessive symbols
- unicode noise
- encoded characters

Examples:
- $$$
- ###
- unicode spam obfuscation

---

### 6. Suspicious Keyword Count
Counts high-risk spam keywords.

Examples:
- free
- win
- money
- urgent
- offer
- click

These terms are commonly found in:
- phishing emails
- scam campaigns
- promotional spam

---

## Hybrid Feature Engineering

After generating manual numerical features,
we combine them with TF-IDF text vectors.

This creates a hybrid feature matrix containing:

- sparse NLP features (TF-IDF)
- dense numerical threat-intelligence features

This hybrid approach is widely used in:
- spam detection systems
- cybersecurity NLP
- email threat intelligence pipelines
- production machine learning systems

---

## Feature Scaling

Manual numerical features are scaled using
StandardScaler before combining them with TF-IDF.

Scaling prevents large-valued features from
dominating model learning and improves
optimization stability for machine learning models.

---

## Final Output

The final dataset contains:
- cleaned NLP text
- engineered spam intelligence features
- scaled numerical signals
- hybrid ML-ready feature representations

These features will be used in the next stage
for advanced model training and evaluation.

In [142]:
# =========================================================
# IMPORT LIBRARIES
# =========================================================

import re

import pandas as pd
import numpy as np

from scipy.sparse import hstack

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler



In [143]:
# =========================================================
# LOAD CLEANED DATASET
# =========================================================

DATA_PATH = (

    "../data/interim/"
    "emails_cleaned.csv"
)

df = pd.read_csv(

    DATA_PATH,

    keep_default_na=False
)

print(

    df.shape
)

df.head()

(5755, 10)


,label,subject,from,to,date,content_type,body,email_length,clean_text,clean_text_length
0,ham,Re: New Sequences Window,Robert Elz <kre@munnari.OZ.AU>,Chris Garrigues <cwg-dated-1030377287.06fa6d@D...,"Thu, 22 Aug 2002 18:26:25 +0700","text/plain; charset=""us-ascii""","Date: Wed, 21 Aug 2002 10:54:46 -0500\n...",1598,date wed aug chri garrigu messageid cant repro...,808
1,ham,[zzzzteana] RE: Alexander,Steve Burt <Steve_Burt@cursor-system.com>,"""'zzzzteana@yahoogroups.com'"" <zzzzteana@yahoo...","Thu, 22 Aug 2002 12:46:18 +0100","text/plain; charset=""US-ASCII""","Martin A posted:\nTassos Papadopoulos, the Gre...",894,martin post tasso papadopoulo greek sculptor b...,399
2,ham,[zzzzteana] Moscow bomber,Tim Chapman <timc@2ubh.com>,zzzzteana <zzzzteana@yahoogroups.com>,"Thu, 22 Aug 2002 13:52:38 +0100","text/plain; charset=""US-ASCII""",Man Threatens Explosion In Moscow \n\nThursday...,1746,man threaten explos moscow thursday august pm ...,902
3,ham,[IRR] Klez: The Virus That Won't Die,Monty Solomon <monty@roscom.com>,undisclosed-recipient:;,"Thu, 22 Aug 2002 09:15:25 -0400","text/plain; charset=""us-ascii""",Klez: The Virus That Won't Die\n \nAlready the...,1125,klez viru wont die alreadi prolif viru ever kl...,590
4,ham,Re: [zzzzteana] Nothing like mama used to make,Stewart Smith <Stewart.Smith@ee.ed.ac.uk>,zzzzteana@yahoogroups.com,"Thu, 22 Aug 2002 14:38:22 +0100","text/plain; charset=""US-ASCII""","> in adding cream to spaghetti carbonara, whi...",1047,ad cream spaghetti carbonara effect pasta make...,464


In [144]:
# =========================================================
# URL COUNT FEATURE
# =========================================================

def count_urls(text):

    urls = re.findall(

        r"http[s]?://\S+|www\.\S+",

        str(text)
    )

    return len(urls)

df["url_count"] = (

    df["body"]

    .apply(count_urls)
)

df[

    [
        "url_count"
    ]

].head()

,url_count
0,1
1,2
2,2
3,2
4,3


In [145]:
# =========================================================
# EXCLAMATION COUNT FEATURE
# =========================================================

def count_exclamations(text):

    return str(text).count("!")

df["exclamation_count"] = (

    df["body"]

    .apply(count_exclamations)
)

df[

    [
        "exclamation_count"
    ]

].head()

,exclamation_count
0,0
1,2
2,2
3,0
4,2


In [146]:
# =========================================================
# UPPERCASE RATIO FEATURE
# =========================================================

def uppercase_ratio(text):

    text = str(text)

    if len(text) == 0:

        return 0

    upper_count = sum(

        1

        for char in text

        if char.isupper()
    )

    return upper_count / len(text)

df["uppercase_ratio"] = (

    df["body"]

    .apply(uppercase_ratio)
)

df[

    [
        "uppercase_ratio"
    ]

].head()

,uppercase_ratio
0,0.028160
1,0.048098
2,0.054983
3,0.037333
4,0.041070


In [147]:
# =========================================================
# HTML TAG COUNT FEATURE
# =========================================================

def count_html_tags(text):

    tags = re.findall(

        r"<[^>]+>",

        str(text)
    )

    return len(tags)

df["html_tag_count"] = (

    df["body"]

    .apply(count_html_tags)
)

df[

    [
        "html_tag_count"
    ]

].head()

,html_tag_count
0,2
1,0
2,0
3,0
4,0


In [148]:
# =========================================================
# SPECIAL CHARACTER COUNT FEATURE
# =========================================================

def count_special_chars(text):

    special_chars = re.findall(

        r"[^a-zA-Z0-9\s]",

        str(text)
    )

    return len(special_chars)

df["special_char_count"] = (

    df["body"]

    .apply(count_special_chars)
)

df[

    [
        "special_char_count"
    ]

].head()

,special_char_count
0,202
1,185
2,182
3,92
4,177


In [149]:
# =========================================================
# SUSPICIOUS WORD COUNT FEATURE
# =========================================================

spam_keywords = [

    "free",
    "win",
    "winner",
    "money",
    "offer",
    "urgent",
    "click",
    "buy",
    "cash",
    "prize"
]

def count_spam_keywords(text):

    text = str(text).lower()

    count = 0

    for word in spam_keywords:

        count += text.count(word)

    return count

df["spam_keyword_count"] = (

    df["body"]

    .apply(count_spam_keywords)
)

df[

    [
        "spam_keyword_count"
    ]

].head()

,spam_keyword_count
0,0
1,2
2,2
3,0
4,2


In [150]:
# =========================================================
# FEATURE OVERVIEW
# =========================================================

feature_columns = [

    "url_count",

    "exclamation_count",

    "uppercase_ratio",

    "html_tag_count",

    "special_char_count",

    "spam_keyword_count"
]

df[

    feature_columns

].describe()

,url_count,exclamation_count,uppercase_ratio,html_tag_count,special_char_count,spam_keyword_count
count,5755.000000,5755.000000,5755.000000,5755.000000,5755.000000,5755.000000
mean,2.737446,2.406603,0.055609,23.871069,261.108254,2.730495
std,6.306039,7.017018,0.056189,86.657367,534.028631,6.847398
min,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000
25%,1.000000,0.000000,0.029285,0.000000,50.000000,0.000000
50%,2.000000,0.000000,0.040553,0.000000,105.000000,1.000000
75%,3.000000,2.000000,0.059888,2.000000,226.500000,3.000000
max,218.000000,92.000000,0.796758,2121.000000,13138.000000,120.000000


In [151]:
# =========================================================
# SAVE FEATURE COLUMN NAMES
# =========================================================

advanced_feature_columns = [

    "url_count",

    "exclamation_count",

    "uppercase_ratio",

    "html_tag_count",

    "special_char_count",

    "spam_keyword_count"
]

print(

    advanced_feature_columns
)

['url_count', 'exclamation_count', 'uppercase_ratio', 'html_tag_count', 'special_char_count', 'spam_keyword_count']


In [152]:
# =========================================================
# LABEL ENCODING
# =========================================================

advanced_label_encoder = LabelEncoder()

df["advanced_label_encoded"] = (

    advanced_label_encoder.fit_transform(
        df["label"]
    )
)

df[

    [
        "label",
        "advanced_label_encoded"
    ]

].head()

,label,advanced_label_encoded
0,ham,0
1,ham,0
2,ham,0
3,ham,0
4,ham,0


In [153]:
# =========================================================
# DEFINE ADVANCED FEATURES
# =========================================================

advanced_text_features = df["clean_text"]

advanced_manual_features = df[

    [

        "url_count",

        "exclamation_count",

        "uppercase_ratio",

        "html_tag_count",

        "special_char_count",

        "spam_keyword_count"
    ]
]

advanced_target = df["advanced_label_encoded"]

In [154]:
# =========================================================
# ADVANCED TRAIN TEST SPLIT
# =========================================================

(
    X_train_text_advanced,
    X_test_text_advanced,

    X_train_manual_advanced,
    X_test_manual_advanced,

    y_train_advanced,
    y_test_advanced

) = train_test_split(

    advanced_text_features,

    advanced_manual_features,

    advanced_target,

    test_size=0.2,

    random_state=42,

    stratify=advanced_target
)

print(

    X_train_text_advanced.shape
)

print(

    X_test_text_advanced.shape
)

(4604,)
(1151,)


In [155]:
# =========================================================
# ADVANCED TF-IDF VECTORIZATION
# =========================================================

advanced_vectorizer = TfidfVectorizer(

    max_features=5000,

    ngram_range=(1, 2),

    min_df=2,

    sublinear_tf=True
)

X_train_tfidf_advanced = (

    advanced_vectorizer.fit_transform(

        X_train_text_advanced
    )
)

X_test_tfidf_advanced = (

    advanced_vectorizer.transform(

        X_test_text_advanced
    )
)

print(

    X_train_tfidf_advanced.shape
)

print(

    X_test_tfidf_advanced.shape
)

(4604, 5000)
(1151, 5000)


In [156]:
# =========================================================
# SCALE ADVANCED MANUAL FEATURES
# =========================================================

advanced_scaler = StandardScaler()

X_train_manual_scaled_advanced = (

    advanced_scaler.fit_transform(

        X_train_manual_advanced
    )
)

X_test_manual_scaled_advanced = (

    advanced_scaler.transform(

        X_test_manual_advanced
    )
)

In [157]:
# =========================================================
# COMBINE ADVANCED FEATURES
# =========================================================

X_train_combined_advanced = hstack([

    X_train_tfidf_advanced,

    X_train_manual_scaled_advanced
]).tocsr()

X_test_combined_advanced = hstack([

    X_test_tfidf_advanced,

    X_test_manual_scaled_advanced
]).tocsr()

print(

    X_train_combined_advanced.shape
)

print(

    X_test_combined_advanced.shape
)

print(

    type(X_train_combined_advanced)
)

(4604, 5006)
(1151, 5006)
<class 'scipy.sparse._csr.csr_matrix'>


In [158]:
print(

    type(X_train_combined_advanced)
)

<class 'scipy.sparse._csr.csr_matrix'>


In [159]:
# =========================================================
# CREATE ARTIFACT DIRECTORIES
# =========================================================

import os

os.makedirs(

    "../artifacts",

    exist_ok=True
)

In [160]:
# =========================================================
# SAVE ADVANCED SCALER
# =========================================================

import joblib

advanced_scaler_path = (

    "../artifacts/"
    "advanced_manual_feature_scaler.pkl"
)

joblib.dump(

    advanced_scaler,
    advanced_scaler_path
)

print(

    f"Advanced scaler saved to:\n{advanced_scaler_path}"
)

Advanced scaler saved to:
../artifacts/advanced_manual_feature_scaler.pkl


In [161]:
# =========================================================
# SAVE ADVANCED TF-IDF VECTORIZER
# =========================================================

advanced_vectorizer_path = (

    "../artifacts/"
    "advanced_hybrid_tfidf_vectorizer.pkl"
)

joblib.dump(

    advanced_vectorizer,
    advanced_vectorizer_path
)

print(

    f"Advanced vectorizer saved to:\n{advanced_vectorizer_path}"
)

Advanced vectorizer saved to:
../artifacts/advanced_hybrid_tfidf_vectorizer.pkl


In [162]:
# =========================================================
# SAVE ADVANCED LABEL ENCODER
# =========================================================

advanced_label_encoder_path = (

    "../artifacts/"
    "advanced_hybrid_label_encoder.pkl"
)

joblib.dump(

    advanced_label_encoder,
    advanced_label_encoder_path
)

print(

    f"Advanced label encoder saved to:\n{advanced_label_encoder_path}"
)

Advanced label encoder saved to:
../artifacts/advanced_hybrid_label_encoder.pkl


In [163]:
# =========================================================
# SAVE FEATURE COLUMN METADATA
# =========================================================

feature_columns_path = (

    "../artifacts/"
    "advanced_feature_columns.pkl"
)

joblib.dump(

    advanced_feature_columns,
    feature_columns_path
)

print(

    f"Feature columns saved to:\n{feature_columns_path}"
)

Feature columns saved to:
../artifacts/advanced_feature_columns.pkl


In [164]:
# =========================================================
# EXPORT ADVANCED FEATURE ENGINEERED DATASET
# =========================================================

advanced_feature_dataset_path = (

    "../data/feature_engineered/"
    "advanced_emails_feature_engineered.csv"
)

os.makedirs(

    "../data/feature_engineered",

    exist_ok=True
)

df.to_csv(

    advanced_feature_dataset_path,

    index=False
)

print(

    "Advanced feature engineered dataset saved to:\n"
    f"{advanced_feature_dataset_path}"
)

Advanced feature engineered dataset saved to:
../data/feature_engineered/advanced_emails_feature_engineered.csv


# =========================================================
# HYBRID FEATURE MATRIX
# =========================================================

Final feature matrix contains:

- 5000 TF-IDF textual features
- 6 engineered security features

Total Features:
5006

This hybrid representation combines:
- NLP semantic understanding
- email threat intelligence signals
- behavioral spam indicators